# Semana 21 · ¿Por qué hay mareas?

Este cuaderno acompaña a la sesión de proyecto de la semana 21: la aproximación binomial, el campo de mareas alrededor de la Tierra y un modelo sencillo de mareas vivas y muertas.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. ¿Cuánto se equivoca la aproximación binomial?

Comparamos $(1 + x)^n$ con $1 + nx$ para varios valores de $x$ y $n = -2$, el caso que usamos para las mareas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = -2
print("    x      exacto      aproximado   error relativo")
for x in [0.5, 0.1, 0.05, 0.017, 0.001]:
    exacto = (1 + x) ** n
    aprox = 1 + n * x
    print(f"{x:6.3f}   {exacto:10.6f}   {aprox:10.6f}    {abs(aprox - exacto) / exacto:9.2e}")

Con $x = 0{,}5$ la aproximación es mala; con $x = 0{,}017$ (el radio de la Tierra dividido por la distancia a la Luna), el error es de una parte en mil. Cada vez que $x$ se divide por 10, el error se divide por unos 100: es del orden de $x^2$.

## 2. La aceleración de marea, exacta y aproximada

En el punto de la Tierra más cercano a la Luna, comparamos la diferencia exacta $GM/(d-R)^2 - GM/d^2$ con la aproximación $2GMR/d^3$.

In [ ]:
GM_L = 4.90e12      # m³/s², Luna
GM_S = 1.327e20     # m³/s², Sol
R = 6.371e6         # m
d_L = 3.844e8       # m
d_S = 1.496e11      # m

for nombre, GM, d in [("Luna", GM_L, d_L), ("Sol", GM_S, d_S)]:
    exacta = GM / (d - R)**2 - GM / d**2
    aprox = 2 * GM * R / d**3
    print(f"{nombre}: atracción en el centro de la Tierra = {GM / d**2:.3e} m/s²   marea exacta = {exacta:.3e}   aproximada = {aprox:.3e} m/s²")

print(f"\nEl Sol atrae a la Tierra {GM_S / d_S**2 / (GM_L / d_L**2):.0f} veces más que la Luna,")
print(f"pero la marea lunar es {(GM_L / d_L**3) / (GM_S / d_S**3):.2f} veces la solar.")

## 3. El campo de mareas alrededor de la Tierra

Para cada punto de la superficie calculamos la aceleración que produce la Luna **menos** la que produce en el centro de la Tierra. Esa diferencia es lo que nota el agua respecto a la Tierra. La Luna está muy a la derecha, sobre el eje $x$. Dibujamos las flechas con `plt.quiver`.

In [ ]:
def acel_luna(p):
    """Aceleración que produce la Luna en el punto p (vector en m). La Luna está en (d_L, 0)."""
    luna = np.array([d_L, 0.0])
    dvec = luna - p
    return GM_L / np.linalg.norm(dvec)**3 * dvec

centro = acel_luna(np.array([0.0, 0.0]))
angulos = np.radians(np.arange(0, 360, 15))
puntos = R * np.column_stack([np.cos(angulos), np.sin(angulos)])
marea = np.array([acel_luna(p) - centro for p in puntos])

fig, ax = plt.subplots(figsize=(6, 6))
ax.add_patch(plt.Circle((0, 0), R, color="lightblue"))
ax.quiver(puntos[:, 0], puntos[:, 1], marea[:, 0], marea[:, 1], angles="xy", scale_units="xy", scale=3e-13, color="navy")
ax.annotate("hacia la Luna →", (R * 1.15, R * 0.3))
ax.set_aspect("equal"); ax.set_xlim(-2 * R, 2 * R); ax.set_ylim(-2 * R, 2 * R); ax.axis("off")
ax.set_title("Aceleración de marea de la Luna sobre la superficie terrestre")
plt.show()

Las flechas apuntan hacia fuera en los dos lados alineados con la Luna (el cercano y el lejano) y hacia dentro en los lados perpendiculares. El agua se acumula en dos abultamientos.

**Ejercicio.** Comprueba que, en el lado lejano, la flecha mide casi lo mismo que en el lado cercano. Calcula los dos módulos y su cociente.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
cerca = np.linalg.norm(acel_luna(np.array([R, 0.0])) - centro)
lejos = np.linalg.norm(acel_luna(np.array([-R, 0.0])) - centro)
print(f"lado cercano: {cerca:.4e} m/s²   lado lejano: {lejos:.4e} m/s²   cociente: {cerca / lejos:.3f}")

Son casi iguales: el cercano es un 5 % mayor. La aproximación $2GMR/d^3$ da el mismo valor para los dos; la diferencia es el término de orden $x^2$ que hemos despreciado.

## 4. Un modelo de mareas vivas y muertas

La marea en un puerto se parece a la suma de ondas, cada una con su periodo. Las dos principales son:

- la **lunar** (los oceanógrafos la llaman M2), con periodo de 12,42 horas: medio día lunar;
- la **solar** (S2), con periodo de 12 horas exactas: medio día solar, y con algo menos de la mitad de amplitud (usamos 0,46 veces la lunar, la proporción teórica).

Cuando las dos coinciden en fase, se suman (mareas vivas); cuando están en oposición, se restan (mareas muertas). Como sus periodos son un poco distintos, pasan de una situación a otra poco a poco.

In [ ]:
T_M2, T_S2 = 12.4206, 12.0      # horas
A_M2, A_S2 = 1.0, 0.46          # amplitudes relativas
t = np.arange(0, 30 * 24, 0.1)  # 30 días, en horas
nivel = A_M2 * np.cos(2 * np.pi * t / T_M2) + A_S2 * np.cos(2 * np.pi * t / T_S2)

plt.figure(figsize=(12, 3.5))
plt.plot(t / 24, nivel, lw=0.7)
plt.xlabel("días"); plt.ylabel("nivel del mar (relativo)")
plt.title("Suma de la marea lunar y la solar")
plt.grid(alpha=0.3); plt.show()

**Ejercicio.** El ciclo de vivas y muertas se repite cuando la onda lunar le ha sacado una oscilación completa a la solar. Con la fórmula del periodo sinódico de la semana 13, $1/P = 1/T_{S2} - 1/T_{M2}$, calcula ese periodo en días. Después mídelo en la gráfica: busca los días con la mayor amplitud y calcula el tiempo entre ellos.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
P = 1 / (1 / T_S2 - 1 / T_M2)
print(f"Periodo del ciclo de vivas y muertas: {P:.1f} h = {P / 24:.2f} días")

# Medida en la gráfica: amplitud de cada día (máximo menos mínimo) y días con amplitud máxima
dias = np.arange(30)
amplitud = np.array([np.ptp(nivel[(t >= 24 * k) & (t < 24 * (k + 1))]) for k in dias])
maximos = [int(k) for k in dias if amplitud[k] >= amplitud[max(k - 1, 0)] and amplitud[k] >= amplitud[min(k + 1, 29)]]
print("Días con mareas vivas en el modelo:", maximos)

plt.bar(dias, amplitud)
plt.xlabel("día"); plt.ylabel("amplitud diaria"); plt.title("Mareas vivas y muertas")
plt.show()

Sale 14,77 días: medio mes sinódico (29,53 días), el tiempo entre luna nueva y luna llena. Es la explicación numérica de por qué las mareas vivas coinciden con esas fases.

## 5. Experimenta

1. Cambia la amplitud solar a 0 y comprueba que desaparece el ciclo de vivas y muertas.
2. ¿Cuánto vale la amplitud en mareas vivas y en mareas muertas en el modelo? Comprueba que son $A_{M2} + A_{S2}$ y $A_{M2} - A_{S2}$.
3. Las mareas reales tienen más componentes. Añade una onda diurna (K1, periodo de 23,93 horas, amplitud 0,3) y mira cómo las dos pleamares de cada día dejan de ser iguales: es la **desigualdad diurna**, muy marcada en algunas costas.

In [ ]:
# Tu experimento aquí